# GAT-LSTM v01 — shared selected features

GAT-LSTM dùng cùng 10 node features, 3 edge features, GAT encoder, service embedding và prediction head của GAT-GRU. Khác biệt duy nhất trong graph-temporal backbone là thay GRU bằng LSTM.

Quy trình: train trên train, chọn checkpoint và hyperparameters bằng validation, sau đó mới đánh giá 5 seed trên frozen test.

In [1]:
import sys
from pathlib import Path

import torch

ROOT = Path.cwd().resolve()
while not (ROOT / 'modeling-common').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'modeling-common'))
import forecast_ablation as fa

ctx = fa.load_context(ROOT)
selection = fa.load_selection_contract(
    ROOT / 'configs' / 'gatgru' / 'feature_contract_v01.json',
    require_edge=True,
)
base_cfg = fa.load_baseline_config(ctx)
base_cfg['use_edge_features'] = True
base_cfg['emb_dim'] = 4
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT = (ROOT / 'artifacts' / 'gatlstm' /
       'v01_shared_selected_features')

print('node features:', selection['node_features'])
print('edge features:', selection['edge_features'])
print('model kind: gatlstm')
print('device:', DEVICE)
print('output:', OUT)

node features: ['rps_in', 'rps_in_delta', 'rps_per_replica', 'latency_p50_ms', 'latency_p95_ms', 'cpu_cores', 'mem_mib', 'net_rx_kBps', 'net_tx_kBps', 'replicas']
edge features: ['call_rate', 'call_ratio', 'edge_latency_p95_ms']
model kind: gatlstm
device: cpu
output: F:\0. UIT\KLTN\thesis-gnn-rl-autoscaling\math-load-testing\experiments\gatlstm\v01_shared_selected_features


## 1. Tune training hyperparameters trên validation

Graph topology, GAT architecture, LSTM hidden size, feature contract và embedding được khóa. Test chưa được truy cập ở bước này.

In [2]:
best_cfg, study = fa.run_optuna_fixed_architecture(
    ctx,
    'gatlstm',
    selection['node_features'],
    selection['edge_features'],
    base_cfg,
    OUT / 'tuning',
    n_trials=24,
    seed=42,
    device=DEVICE,
    score_weights=fa.CPU_DIAGNOSTIC_WEIGHTS,
    parent_version='gatgru_v05_selected_features_final',
)
assert best_cfg['use_edge_features'] is True
print('best validation score:', study.best_value)
best_cfg

[I 2026-10-26 09:20:17,277] A new study created in RDB with name: gatlstm_fb5407310c2dfc0d_fixed_arch
[I 2026-10-26 09:44:07,466] Trial 0 finished with value: 0.1032001654993075 and parameters: {'window': 18, 'dropout': 0.11973169683940732, 'lr': 0.0006427207987637223, 'weight_decay': 2.9375384576328313e-06, 'batch_size': 64}. Best is trial 0 with value: 0.1032001654993075.
[I 2026-10-26 10:18:45,435] Trial 1 finished with value: 0.10497373204402463 and parameters: {'window': 24, 'dropout': 0.16648852816008436, 'lr': 0.0007037018186923737, 'weight_decay': 3.5113563139704077e-06, 'batch_size': 128}. Best is trial 0 with value: 0.1032001654993075.
[I 2026-10-26 10:34:57,316] Trial 2 finished with value: 0.10452663283342953 and parameters: {'window': 24, 'dropout': 0.027898772130408367, 'lr': 0.0008001480489146253, 'weight_decay': 1.2562773503807034e-05, 'batch_size': 64}. Best is trial 0 with value: 0.1032001654993075.
[I 2026-10-26 10:56:07,652] Trial 3 finished with value: 0.1027139894

best validation score: 0.10271398943128791


{'window': 18,
 'dropout': 0.12150897038028768,
 'lr': 0.0006579020684258309,
 'batch_size': 64,
 'gat_hidden': 32,
 'heads': 4,
 'gat_layers': 2,
 'gru_hidden': 128,
 'edge_direction': 'both',
 'use_edge_features': True,
 'emb_dim': 4,
 'weight_decay': 1.5673095467235414e-06}

## 2. Huấn luyện 5 seed và đánh giá frozen test

Mỗi seed huấn luyện trên train và dùng validation để chọn checkpoint. `run_final` không refit trên train + validation.

In [3]:
prediction_path, training_meta = fa.run_final(
    ctx,
    'gatlstm',
    selection['node_features'],
    selection['edge_features'],
    best_cfg,
    OUT,
    seeds=(0, 1, 2, 3, 4),
    device=DEVICE,
    model_name='gatlstm_selected',
    parent_version='gatgru_v05_selected_features_final',
)
assert training_meta['model_kind'] == 'gatlstm'
print(prediction_path)
training_meta

F:\0. UIT\KLTN\thesis-gnn-rl-autoscaling\math-load-testing\experiments\gatlstm\v01_shared_selected_features\gatlstm_selected_predictions_test.npz


{'model': 'gatlstm_selected',
 'model_kind': 'gatlstm',
 'config': {'window': 18,
  'dropout': 0.12150897038028768,
  'lr': 0.0006579020684258309,
  'batch_size': 64,
  'gat_hidden': 32,
  'heads': 4,
  'gat_layers': 2,
  'gru_hidden': 128,
  'edge_direction': 'both',
  'use_edge_features': True,
  'emb_dim': 4,
  'weight_decay': 1.5673095467235414e-06},
 'seeds': [0, 1, 2, 3, 4],
 'val_mae_scaled': [0.11138754338026047,
  0.11101716756820679,
  0.11042032390832901,
  0.110395148396492,
  0.11117793619632721],
 'node_features': ['rps_in',
  'rps_in_delta',
  'rps_per_replica',
  'latency_p50_ms',
  'latency_p95_ms',
  'cpu_cores',
  'mem_mib',
  'net_rx_kBps',
  'net_tx_kBps',
  'replicas'],
 'edge_features': ['call_rate', 'call_ratio', 'edge_latency_p95_ms'],
 'targets': ['rps_in', 'cpu_cores'],
 'services': ['frontend',
  'adservice',
  'cartservice',
  'checkoutservice',
  'currencyservice',
  'emailservice',
  'paymentservice',
  'productcatalogservice',
  'recommendationservice',
